import os, time, datetime, warnings

import torch
from torch import nn
from torch.optim import lr_scheduler

from datasets.data_manager_STARmap_PLUS import AD_Mouse
from utils.notebook_graph_utils import build_graph_model, state_dict_for_saving
from utils.utils import *
from utils.utils_dataloader import *
from utils.utils_training_STARmap_PLUS import train, test

warnings.filterwarnings("ignore")


In [ ]:
import os, time, datetime, warnings

import torch
from torch import nn
from torch.optim import lr_scheduler

from datasets.data_manager_STARmap_PLUS import AD_Mouse
from utils.notebook_graph_utils import build_graph_model, state_dict_for_saving
from utils.utils import *
from utils.utils_dataloader import *
from utils.utils_training_STARmap_PLUS import train, test

warnings.filterwarnings("ignore")


### Initialize the args and fix seeds

In [ ]:
%run ./args/args_STARmap_PLUS.py
args = args

set_seed(args.seed)
os.environ['CUDA_VISIBLE_DEVICES'] = args.gpu_devices

print("==========\nArgs:{}\n==========".format(args))

### Initialize dataloaders and NicheTrans

In [ ]:
# create the dataset
dataset = AD_Mouse(
    AD_adata_path=args.AD_adata_path,
    Wild_type_adata_path=args.Wild_type_adata_path,
    n_top_genes=args.n_top_genes,
    graph_k=args.graph_k,
    val_ratio=args.val_ratio,
    mask_seed=args.seed,
)
trainloader, testloader, _ = ad_mouse_dataloader(args, dataset)

# create the model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = build_graph_model(args, dataset, use_cell_type=args.use_cell_type).to(device)
print(f"Using device: {device}")


### Initialize loss function (criterion) and optimizer

In [ ]:
criterion = nn.BCEWithLogitsLoss()

if args.optimizer == 'adam':
    optimizer = torch.optim.Adam(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)
elif args.optimizer == 'SGD':
    optimizer = torch.optim.SGD(model.parameters(), lr=args.lr)
else:
    print('unexpected optimizer')

if args.stepsize > 0:
    scheduler = lr_scheduler.StepLR(optimizer, step_size=args.stepsize, gamma=args.gamma)


### Model training and testing

In [ ]:
start_time = time.time()

for epoch in range(args.max_epoch):
    print("==> Epoch {}/{}".format(epoch + 1, args.max_epoch))
    train(model, criterion, optimizer, trainloader, ct_information=True, device=device)
    if args.stepsize > 0:
        scheduler.step()

metrics = test(model, testloader, ct_information=True, device=device)
torch.save(state_dict_for_saving(model), 'WholeSliceGraphTransformer_ct_STARmap_PLUS.pth')

elapsed = round(time.time() - start_time)
elapsed = str(datetime.timedelta(seconds=elapsed))
print("Finished. Total elapsed time (h:m:s): {}".format(elapsed))
